In [1]:
import argparse
import os
import sys
import urllib.request

import ee
import pandas as pd

# ------------------------- CONFIG -------------------------
POINTS_ASSET = "projects/huygens-project/assets/crop_stress/sample_points"  
DRIVE_FOLDER = "crop_stress_exports"

OGUN_MIN_LAT = 6.7            
CROP_FRAC_MIN = 0.5           
ZONE_BREAKS = (7.0, 8.0)
POINTS_PER_ZONE = [3300, 3300, 3400]
SEED = 42

DEFAULT_YEARS = list(range(2001, 2026))
WINDOW_K = list(range(3, 13))  # 16-day windows starting DOY 1 + 16k (k=3 -> DOY 49 ... k=12 -> DOY 193)
QA_MAX = 1                     # SummaryQA: 0 good, 1 marginal
NODATA = -9999                 # fills masked windows so sampleRegions keeps every point


# ------------------------- AOI -------------------------
def build_aoi():
    gaul1 = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))

    def state_geom(name):
        return gaul1.filter(ee.Filter.eq("ADM1_NAME", name)).geometry()

    ogun_north = state_geom("Ogun").intersection(
        ee.Geometry.Rectangle([2.0, OGUN_MIN_LAT, 5.0, 10.0]), ee.ErrorMargin(100)
    )
    aoi = (
        ee.FeatureCollection(
            [ee.Feature(state_geom("Oyo")), ee.Feature(state_geom("Osun")), ee.Feature(ogun_north)]
        )
        .geometry()
        .dissolve(100)
    )
    lgas = (
        ee.FeatureCollection("FAO/GAUL/2015/level2")
        .filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
        .filterBounds(aoi)
    )
    return aoi, lgas


def modis_projection():
    return ee.ImageCollection("MODIS/061/MOD13Q1").first().select("NDVI").projection()


# ------------------------- MODE: points -------------------------
def make_points(aoi, lgas, modis_proj):
    # Cropland fraction per MODIS pixel from WorldCover (class 40 = cropland)
    wc_crop = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map").eq(40)
    frac = (
        wc_crop.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
        .reproject(modis_proj)
        .rename("crop_frac")
    )
    crop_mask = frac.gte(CROP_FRAC_MIN)

    # Latitude zones (proxy strata)
    lat = ee.Image.pixelLonLat().select("latitude")
    zone = (
        ee.Image(1)
        .where(lat.lt(ZONE_BREAKS[0]), 0)
        .where(lat.gte(ZONE_BREAKS[1]), 2)
        .rename("zone")
        .toInt()
    )
    strat = zone.addBands(frac).updateMask(crop_mask)

    pts = strat.stratifiedSample(
        numPoints=0,
        classBand="zone",
        region=aoi,
        projection=modis_proj,
        seed=SEED,
        classValues=[0, 1, 2],
        classPoints=POINTS_PER_ZONE,
        dropNulls=True,
        tileScale=4,
        geometries=True,
    )

    # Attach LGA name + point id
    def add_ids(f):
        lga = lgas.filterBounds(f.geometry()).first()
        name = ee.Algorithms.If(lga, ee.Feature(lga).get("ADM2_NAME"), "NA")
        return f.set({"lga": name, "pid": f.get("system:index")})

    pts = pts.map(add_ids)

    print("Points per zone:", pts.aggregate_histogram("zone").getInfo())
    print("Total points:", pts.size().getInfo())

    task = ee.batch.Export.table.toAsset(
        collection=pts, description="sample_points", assetId=POINTS_ASSET
    )
    task.start()
    print(f"Started asset export -> {POINTS_ASSET}  (task id: {task.id})")


# ------------------------- MODE: ndvi -------------------------
def clean_ndvi_collection():
    mod = ee.ImageCollection("MODIS/061/MOD13Q1")
    myd = ee.ImageCollection("MODIS/061/MYD13Q1")
    ic = mod.merge(myd).select(["NDVI", "SummaryQA"])

    def mask_scale(img):
        ok = img.select("SummaryQA").lte(QA_MAX)
        ndvi = img.select("NDVI").multiply(0.0001).updateMask(ok).rename("ndvi")
        return ndvi.copyProperties(img, ["system:time_start"])

    return ic.map(mask_scale)


def year_image(clean, year):
    """Mean NDVI and valid-obs count per 16-day window, as one multi-band image."""
    out = None
    for k in WINDOW_K:
        start = ee.Date.fromYMD(year, 1, 1).advance(k * 16, "day")
        end = start.advance(16, "day")
        w = clean.filterDate(start, end)
        bands = w.mean().rename(f"ndvi_k{k:02d}").addBands(w.count().rename(f"n_k{k:02d}"))
        out = bands if out is None else out.addBands(bands)
    return out.unmask(NODATA)


def export_ndvi(aoi, modis_proj, years):
    points = ee.FeatureCollection(POINTS_ASSET)
    clean = clean_ndvi_collection()

    # Quick preview: mean valid observations per window over the AOI (2020)
    prev = year_image(clean, 2020).select("n_k.*")
    stats = prev.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=250, maxPixels=int(1e9), bestEffort=True, tileScale=4
    )
    print("Preview 2020, mean valid obs per window:", stats.getInfo())

    for y in years:
        img = year_image(clean, y)
        table = img.sampleRegions(
            collection=points, projection=modis_proj, tileScale=4, geometries=False
        ).map(lambda f, y=y: f.set("year", y))

        task = ee.batch.Export.table.toDrive(
            collection=table,
            description=f"modis_ndvi_{y}",
            folder=DRIVE_FOLDER,
            fileFormat="CSV",
        )
        task.start()
        print(f"Started export for {y} (task id: {task.id})")


def download_ndvi_local(modis_proj, years, out_dir, n_chunks=10):
    """Download yearly CSVs straight to a local folder (chunked to stay under EE size limits).
    Resumable: finished years and finished chunks are skipped."""
    os.makedirs(out_dir, exist_ok=True)
    points = ee.FeatureCollection(POINTS_ASSET).randomColumn("rnd", SEED)
    clean = clean_ndvi_collection()
    cols = (
        ["pid", "lga", "zone", "year"]
        + [f"ndvi_k{k:02d}" for k in WINDOW_K]
        + [f"n_k{k:02d}" for k in WINDOW_K]
    )
    for y in years:
        final = os.path.join(out_dir, f"modis_ndvi_{y}.csv")
        if os.path.exists(final):
            print(f"{y}: already done, skipping")
            continue
        img = year_image(clean, y)
        parts = []
        for i in range(n_chunks):
            part = os.path.join(out_dir, f"_part_{y}_{i}.csv")
            if not os.path.exists(part):
                chunk = points.filter(
                    ee.Filter.And(ee.Filter.gte("rnd", i / n_chunks), ee.Filter.lt("rnd", (i + 1) / n_chunks))
                )
                tbl = img.sampleRegions(
                    collection=chunk, projection=modis_proj, tileScale=4, geometries=False
                ).map(lambda f, y=y: f.set("year", y))
                url = tbl.getDownloadURL(filetype="csv", selectors=cols)
                urllib.request.urlretrieve(url, part)
            parts.append(part)
        df = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
        df = df.replace(NODATA, float("nan"))
        df.to_csv(final, index=False)
        for p in parts:
            os.remove(p)
        print(f"{y}: saved {len(df)} rows -> {final}")


# ------------------------- main -------------------------
def main(argv=None):
    """In Jupyter call e.g. main(["points", "--project", "my-project"])."""
    ap = argparse.ArgumentParser()
    ap.add_argument("mode", choices=["points", "ndvi"])
    ap.add_argument("--project", required=True, help="Google Cloud project registered for Earth Engine")
    ap.add_argument("--years", nargs="*", type=int, default=DEFAULT_YEARS)
    ap.add_argument("--local-dir", default=None, help="Download CSVs to this local folder instead of Google Drive")
    args = ap.parse_args(argv)

    try:
        ee.Initialize(project=args.project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=args.project)

    aoi, lgas = build_aoi()
    modis_proj = modis_projection()

    if args.mode == "points":
        make_points(aoi, lgas, modis_proj)
    elif args.local_dir:
        download_ndvi_local(modis_proj, args.years, args.local_dir)
    else:
        export_ndvi(aoi, modis_proj, args.years)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [3]:
main(["ndvi", "--project", "huygens-project", "--local-dir", r"crop_stress\raw"])

2001: already done, skipping
2002: already done, skipping
2003: already done, skipping
2004: already done, skipping
2005: already done, skipping
2006: already done, skipping
2007: already done, skipping
2008: already done, skipping
2009: already done, skipping
2010: already done, skipping
2011: already done, skipping
2012: already done, skipping
2013: already done, skipping
2014: already done, skipping
2015: already done, skipping
2016: already done, skipping
2017: already done, skipping
2018: already done, skipping
2019: already done, skipping
2020: already done, skipping
2021: already done, skipping
2022: already done, skipping
2023: already done, skipping
2024: already done, skipping
2025: already done, skipping


C:\Users\DELL\anaconda3\envs\geospatial\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)
C:\Users\DELL\anaconda3\envs\geospatial\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)
